# Split PCA Dataset into Train / Validation / Test Sets

This notebook splits the PCA-compressed SCR dataset into training, validation, and test sets.

From the PCA compression notebook, the full SCR output dataset was approximated as

$$
Y \approx \mathbf{1}\bar{y}^T + U\Phi,
$$

where:

- $Y \in \mathbb{R}^{N_s \times N_t}$ contains the full NOx output time signals,
- $U \in \mathbb{R}^{N_s \times r}$ contains the PCA coefficients,
- $\Phi \in \mathbb{R}^{r \times N_t}$ contains the PCA modes,
- $\bar{y} \in \mathbb{R}^{N_t}$ is the mean signal.

Here, the sampled kinetic parameter matrix is

$$
A \in \mathbb{R}^{N_s \times 6},
$$

and the PCA coefficient matrix is

$$
U \in \mathbb{R}^{N_s \times r}.
$$

This notebook loads:

- the sampled kinetic parameter matrix $A$ from `lhs_sampling.npy`,
- the PCA-compressed output matrix $U$ from `PCA_Dataset_r20.npz`,

then splits the samples into train, validation, and test sets.

The output of this notebook is a new `.npz` file containing:

- `A_train`, `A_val`, `A_test`,
- `U_train`, `U_val`, `U_test`,
- `idx_train`, `idx_val`, `idx_test`,
- `Phi`, `Y_mean`, and `t`.

## 1. Imports and Paths

This section defines the project paths and checks that the required files exist.

The notebook is assumed to be located in:

```text
~/AME508/2_PCA

In [12]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

# Current notebook is in ~/AME508/2_PCA
ROOT = Path.cwd().parent

DATASET_DIR = ROOT / "1_DATA"
PCA_DIR = ROOT / "2_PCA"

LHS_SAMPLING_PATH = DATASET_DIR / "lhs_sampling.npy"
PCA_DATASET_PATH = PCA_DIR / "PCA_Dataset_r20.npz"
SPLIT_SAVE_PATH = PCA_DIR / "PCA_Dataset_r20_split.npz"

sys.path.append(str(DATASET_DIR))

print("cwd:", Path.cwd())
print("ROOT:", ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("PCA_DIR:", PCA_DIR)

print("\nFiles:")
for name, path in {
    "lhs_sampling.npy": LHS_SAMPLING_PATH,
    "PCA_Dataset_r20.npz": PCA_DATASET_PATH,
}.items():
    print(f"{name:24s} exists = {path.exists()} | {path}")

assert LHS_SAMPLING_PATH.exists(), f"Missing file: {LHS_SAMPLING_PATH}"
assert PCA_DATASET_PATH.exists(), f"Missing file: {PCA_DATASET_PATH}"

cwd: /home/ssariedd/AME508/2_PCA
ROOT: /home/ssariedd/AME508
DATASET_DIR: /home/ssariedd/AME508/1_DATA
PCA_DIR: /home/ssariedd/AME508/2_PCA

Files:
lhs_sampling.npy         exists = True | /home/ssariedd/AME508/1_DATA/lhs_sampling.npy
PCA_Dataset_r20.npz      exists = True | /home/ssariedd/AME508/2_PCA/PCA_Dataset_r20.npz


## 2. Load PCA Dataset and Parameter Samples

Load the kinetic parameter matrix \(A\), PCA coefficient matrix \(U\), and reconstruction data \(\Phi\), \(\bar{y}\), and \(t\).

We check that

$$
A \in \mathbb{R}^{N_s \times 6},
\qquad
U \in \mathbb{R}^{N_s \times r},
$$

have the same number of samples.

In [6]:
A = np.load(LHS_SAMPLING_PATH)

pca_data = np.load(PCA_DATASET_PATH)

U = pca_data["U"]
Phi = pca_data["Phi"]
Y_mean = pca_data["Y_mean"]
t = pca_data["t"]
r = int(pca_data["r"])

print("PCA dataset keys:", pca_data.files)

print("\nShapes:")
print("A:     ", A.shape)
print("U:     ", U.shape)
print("Phi:   ", Phi.shape)
print("Y_mean:", Y_mean.shape)
print("t:     ", t.shape)
print("r:     ", r)

assert A.shape[0] == U.shape[0]
assert U.shape[1] == r
assert Phi.shape == (r, len(t))
assert Y_mean.shape == (len(t),)

PCA dataset keys: ['r', 't', 'Y_mean', 'Phi', 'U', 'Y_rec', 'explained', 'cum_explained', 'full_dataset_path', 'lhs_sampling_path']

Shapes:
A:      (5000, 6)
U:      (5000, 20)
Phi:    (20, 11993)
Y_mean: (11993,)
t:      (11993,)
r:      20


## 3. Create Train / Validation / Test Indices

Randomly split the \(N_s\) samples into train, validation, and test sets.

We use an 80/10/10 split and save the indices so the split is reproducible.

In [9]:
# Split fractions
train_frac = 0.80
val_frac = 0.10
test_frac = 0.10

seed = 0
rng = np.random.default_rng(seed)

n_samples = A.shape[0]
indices = np.arange(n_samples)
rng.shuffle(indices)

n_train = int(train_frac * n_samples)
n_val = int(val_frac * n_samples)
n_test = n_samples - n_train - n_val

idx_train = indices[:n_train]
idx_val = indices[n_train:n_train + n_val]
idx_test = indices[n_train + n_val:]

print("n_samples:", n_samples)
print("n_train:  ", len(idx_train))
print("n_val:    ", len(idx_val))
print("n_test:   ", len(idx_test))

assert len(idx_train) + len(idx_val) + len(idx_test) == n_samples
assert len(set(idx_train) & set(idx_val)) == 0
assert len(set(idx_train) & set(idx_test)) == 0
assert len(set(idx_val) & set(idx_test)) == 0

n_samples: 5000
n_train:   4000
n_val:     500
n_test:    500


## 4. Apply the Split

Use the train, validation, and test indices to split the kinetic parameter matrix \(A\) and PCA coefficient matrix \(U\).

In [10]:
A_train = A[idx_train]
A_val = A[idx_val]
A_test = A[idx_test]

U_train = U[idx_train]
U_val = U[idx_val]
U_test = U[idx_test]

print("A_train:", A_train.shape)
print("A_val:  ", A_val.shape)
print("A_test: ", A_test.shape)

print("\nU_train:", U_train.shape)
print("U_val:  ", U_val.shape)
print("U_test: ", U_test.shape)

assert A_train.shape[0] == U_train.shape[0]
assert A_val.shape[0] == U_val.shape[0]
assert A_test.shape[0] == U_test.shape[0]

assert A_train.shape[1] == 6
assert U_train.shape[1] == r

A_train: (4000, 6)
A_val:   (500, 6)
A_test:  (500, 6)

U_train: (4000, 20)
U_val:   (500, 20)
U_test:  (500, 20)


## 5. Save Split Dataset

Save the train, validation, and test arrays, along with the split indices and PCA reconstruction data.

In [11]:
np.savez(
    SPLIT_SAVE_PATH,
    A_train=A_train,
    A_val=A_val,
    A_test=A_test,
    U_train=U_train,
    U_val=U_val,
    U_test=U_test,
    idx_train=idx_train,
    idx_val=idx_val,
    idx_test=idx_test,
    Phi=Phi,
    Y_mean=Y_mean,
    t=t,
    r=r,
    seed=seed,
    train_frac=train_frac,
    val_frac=val_frac,
    test_frac=test_frac,
)

print("Saved split dataset to:")
print(SPLIT_SAVE_PATH)

Saved split dataset to:
/home/ssariedd/AME508/2_PCA/PCA_Dataset_r20_split.npz
